# Ch 12 — 학습 루프와 AdamW

AdamW 옵티마이저 + cosine schedule + gradient clipping 으로 구성된 표준 5단계 학습 루프를 직접 만들고 sanity check 를 통해 검증한다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part4/ch12_training_loop.ipynb)

In [ ]:
# !pip install -q torch tokenizers datasets

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from dataclasses import dataclass

# Device check
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'device: {device}')
print(f'torch:  {torch.__version__}')

## GPTMini 모델 정의 (Ch 10)

Ch 10 의 `GPTMini` 를 인라인으로 정의한다. 실제 학습에서는 `from nano_gpt import GPTMini, GPTConfig` 로 대체.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 8000
    n_layer: int = 6
    n_head: int = 8
    d_model: int = 320
    max_len: int = 512

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.d_model = cfg.d_model
        self.qkv  = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = int(cfg.d_model * 8 / 3)
        self.gate = nn.Linear(cfg.d_model, h, bias=False)
        self.up   = nn.Linear(cfg.d_model, h, bias=False)
        self.down = nn.Linear(h, cfg.d_model, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.RMSNorm(cfg.d_model)
        self.attn  = CausalSelfAttention(cfg)
        self.norm2 = nn.RMSNorm(cfg.d_model)
        self.ffn   = FFN(cfg)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class GPTMini(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg    = cfg
        self.embed  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos    = nn.Embedding(cfg.max_len, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm   = nn.RMSNorm(cfg.d_model)
        self.head   = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, x, y=None):
        B, T = x.shape
        h = self.embed(x) + self.pos(torch.arange(T, device=x.device))
        for block in self.blocks:
            h = block(h)
        logits = self.head(self.norm(h))
        loss = None
        if y is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size), y.view(-1))
        return logits, loss

cfg   = GPTConfig()
model = GPTMini(cfg).to(device)
print(f'파라미터: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

## 최소 예제 — 50줄 학습 루프

이 6 줄이 **모든 학습 루프의 본체**. nanoGPT, Llama, GPT-4 모두 똑같음 (분산·precision 만 다름).

```python
for batch in loader:
    logits, loss = model(batch.x, batch.y)   # 1. forward
    loss.backward()                          # 2. backward
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # 3. clip
    optimizer.step()                         # 4. parameter 갱신
    optimizer.zero_grad()                    # 5. gradient 초기화
    scheduler.step()                         # 6. learning rate 갱신
```

In [ ]:
# 더미 배치 생성기 (실제 학습에서는 BinLoader 로 대체)
def get_dummy_batch(batch_size=4, seq_len=64, vocab_size=8000):
    x = torch.randint(0, vocab_size, (batch_size, seq_len))
    y = torch.randint(0, vocab_size, (batch_size, seq_len))
    return x.to(device), y.to(device)

In [ ]:
# 1. Optimizer — weight_decay 두 그룹으로 분리
decay_params, no_decay_params = [], []
for n, p in model.named_parameters():
    if p.dim() < 2 or 'norm' in n or 'embed' in n:
        no_decay_params.append(p)
    else:
        decay_params.append(p)

optimizer = torch.optim.AdamW(
    [{"params": decay_params,    "weight_decay": 0.1},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=6e-4, betas=(0.9, 0.95), eps=1e-8,
)

print(f'decay params:    {sum(p.numel() for p in decay_params)/1e6:.2f}M')
print(f'no-decay params: {sum(p.numel() for p in no_decay_params)/1e6:.2f}M')

In [ ]:
# 2. Scheduler — warmup + cosine
total_steps, warmup_steps = 50_000, 1_000

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))  # 0.1 = min/peak ratio

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# lr 곡선 확인
lrs = [lr_lambda(s) * 6e-4 for s in range(2000)]
plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel('step'); plt.ylabel('lr')
plt.title('Warmup + Cosine Schedule (첫 2000 step)')
plt.tight_layout()
plt.show()

In [ ]:
# 3. 학습 루프 본체 (데모: 200 step)
# 실제 학습에서는 total_steps=50_000 으로 변경
model.train()
demo_steps = 200
log_steps, log_losses = [], []

for step in range(demo_steps):
    x, y = get_dummy_batch()

    logits, loss = model(x, y)                                         # 1. forward
    loss.backward()                                                    # 2. backward (gradient 계산)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)            # 3. clip
    optimizer.step()                                                   # 4. parameter 갱신
    scheduler.step()                                                   # 6. learning rate 갱신
    optimizer.zero_grad(set_to_none=True)                              # 5. gradient 초기화

    if step % 50 == 0:
        lr = optimizer.param_groups[0]['lr']
        log_steps.append(step)
        log_losses.append(loss.item())
        print(f'  step {step:5d} | loss {loss.item():.3f} | lr {lr:.5f}')

print(f'\n{demo_steps} step 완료')

## 실전 — 학습 시작 전 Sanity Check

학습 50,000 step 돌리기 전에 **5분 안에** 다음을 확인.

- **5.1** 학습 전 loss ≈ ln(vocab)
- **5.2** 1-batch overfit → loss 0 부근
- **5.3** lr 곡선 모양 확인

In [ ]:
# 5.1 학습 전 loss 확인
model_check = GPTMini(cfg).to(device)  # 미학습 새 모델
model_check.eval()
with torch.no_grad():
    x, y = get_dummy_batch()
    _, loss = model_check(x, y)
print(f'학습 전 loss:   {loss.item():.3f}')
print(f'ln(vocab=8000): {math.log(8000):.3f}')
# 기대: 8.99 부근. 1~2 차이는 OK. 0 또는 100 이면 모델 깨짐.

In [ ]:
# 5.2 1-batch overfit 검증
# 같은 batch 를 100번 학습. loss 가 0 부근으로 떨어지면 모델·loss·optimizer 가 정상.
model_overfit = GPTMini(cfg).to(device)
opt_overfit   = torch.optim.AdamW(model_overfit.parameters(), lr=1e-3)

x, y = get_dummy_batch(batch_size=4, seq_len=32)
losses_overfit = []

model_overfit.train()
for i in range(100):
    _, loss = model_overfit(x, y)
    loss.backward()
    opt_overfit.step()
    opt_overfit.zero_grad()
    losses_overfit.append(loss.item())
    if i % 20 == 0:
        print(f'  step {i}: {loss.item():.3f}')
# 기대: 8.99 → 0.5 미만. 떨어지지 않으면 lr 또는 모델 문제.

In [ ]:
# 5.3 처음 2000 step 에서 lr 곡선 확인
import matplotlib.pyplot as plt
lrs = [lr_lambda(s) * 6e-4 for s in range(2000)]
plt.plot(lrs)
plt.xlabel('step'); plt.ylabel('lr')
plt.title('학습률 곡선 (warmup → cosine)')
plt.show()
# 기대: 0 → 6e-4 (warmup 1000) → cosine decay 시작

## 연습문제

1. 본인 환경에서 §5.2 의 1-batch overfit 을 200 step 돌려라. loss 가 8.99 → ? 까지 떨어지는가? 안 떨어지면 lr 을 `1e-3`, `6e-4`, `3e-4`, `1e-4` 로 바꿔 다시.
2. cosine schedule 의 `min_lr / peak_lr` 비율을 `0.0` / `0.1` / `0.5` 로 바꿔 같은 step 학습 후 평가 loss 비교.
3. `betas=(0.9, 0.999)` (Adam 기본) vs `(0.9, 0.95)` (LLM 표준) 으로 1000 step 학습 후 손실 곡선 비교. 차이가 보이는가?
4. weight_decay 를 `0.0` / `0.1` / `0.5` 로 바꿔 학습 후 마지막 평가 loss + parameter norm (`sum(p.norm() for p in model.parameters())`) 비교.
5. **(생각해볼 것)** warmup 이 필요한 이유는 "학습 초반 gradient 가 크다" 였다. **왜** 학습 초반 gradient 가 큰가? RMSNorm·residual 관점에서.

In [ ]:
# 연습 1: 1-batch overfit, lr 실험


In [ ]:
# 연습 2: min_lr / peak_lr 비율 실험


In [ ]:
# 연습 3: betas 비교


In [ ]:
# 연습 4: weight_decay 비교


In [ ]:
# 연습 5: 자유 답변
